## Analytical Analysis


In [1]:
import pandas as pd

In [2]:
companies = pd.read_csv("../data/processed/companies_clean.csv")
postings = pd.read_csv("../data/processed/postings_clean.csv")
salaries = pd.read_csv("../data/processed/salaries_clean.csv")

In [34]:
job_skills = pd.read_csv("../data/raw/linkedin-job-postings/jobs/job_skills.csv")
skills = pd.read_csv("../data/raw/linkedin-job-postings/mappings/skills.csv")
companies_industries = pd.read_csv("../data/raw/linkedin-job-postings/companies/company_industries.csv")
job_industries = pd.read_csv("../data/raw/linkedin-job-postings/jobs/job_industries.csv")
industries = pd.read_csv("../data/raw/linkedin-job-postings/mappings/industries.csv")
employee_counts = pd.read_csv("../data/raw/linkedin-job-postings/companies/employee_counts.csv")

## Job demand by role

Job demand is approximated by the number of job postings per job title.
Job titles are used as a proxy for roles, acknowledging that they are not
standardized across postings.

JOB DEMAND
    postings_clean -> job_id, title



In [6]:
job_demand_by_role = (
    postings
    .groupby('title')
    .size()
    .reset_index(name='num_postings')
    .sort_values('num_postings', ascending=False)
)

In [7]:
job_demand_by_role.head(10)

,title,num_postings
54901,Sales Manager,673
15069,Customer Service Representative,373
47243,Project Manager,354
2510,Administrative Assistant,254
56343,Senior Accountant,238
21744,Executive Assistant,228
55357,Salesperson,211
50465,Registered Nurse,210
49827,Receptionist,204
64091,Staff Accountant,200


## Skill demand analysis

Skill demand is approximated by counting how often each skill appears across job
postings. This analysis reflects skill popularity rather than proficiency level
or job seniority. To better contextualize skill demand, the most frequent skills were also analyzed
within the most in-demand job roles.

job_skills -> job_id, skill_abr
skills -> skill_abr, skill_name

In [11]:
skills_df = postings_skills.dropna(subset=['skill_name'])

skill_demand = (
    skills_df
    .groupby('skill_name')
    .size()
    .reset_index(name='num_postings')
    .sort_values('num_postings', ascending=False)
)

In [12]:
skill_demand.head(10)

,skill_name,num_postings
16,Information Technology,25256
29,Sales,21193
18,Management,20385
19,Manufacturing,17728
14,Health Care Provider,16675
5,Business Development,13304
11,Engineering,12530
21,Other,12314
12,Finance,8011
20,Marketing,5400


In [13]:
top_roles = (
    postings
    .groupby('title')
    .size()
    .sort_values(ascending=False)
    .head(10)
    .index
)

In [14]:
skills_by_role = (
    skills_df[skills_df['title'].isin(top_roles)]
    .groupby(['title', 'skill_name'])
    .size()
    .reset_index(name='num_postings')
    .sort_values(['title', 'num_postings'], ascending=[True, False])
)


In [16]:
skills_by_role[skills_by_role['title'] == top_roles[0]].head(30)

,title,skill_name,num_postings
107,Sales Manager,Sales,668
95,Sales Manager,Business Development,651
102,Sales Manager,Management,12
109,Sales Manager,Training,3
97,Sales Manager,Customer Service,2
98,Sales Manager,Distribution,2
103,Sales Manager,Manufacturing,2
93,Sales Manager,Accounting/Auditing,1
94,Sales Manager,Advertising,1
96,Sales Manager,Consulting,1


## Remote vs on-site roles

Remote job availability is assessed using the `remote_allowed` flag, which
indicates positions that explicitly allow remote work. Missing values do not
represent on-site roles but rather the absence of explicit remote information.
This analysis highlights industries with the highest number of explicitly remote
job postings.

postings_clean -> job_id, company_id, company_name, title, remote_allowed
companies_industries -> company_id, industry

In [17]:
remote_postings = postings[postings['remote_allowed'] == 1]

remote_with_industry = remote_postings.merge(
    companies_industries,
    on='company_id',
    how='left'
)

remote_by_industry = (
    remote_with_industry
    .groupby('industry')
    .size()
    .reset_index(name='num_remote_postings')
    .sort_values('num_remote_postings', ascending=False)
)

In [18]:
remote_by_industry.head(10)

,industry,num_remote_postings
49,IT Services and IT Consulting,2785
106,Software Development,2499
109,Staffing and Recruiting,2081
35,Financial Services,1199
47,Hospitals and Health Care,745
53,Insurance,469
1,Advertising Services,320
14,Business Consulting and Services,305
11,Biotechnology Research,293
67,Medical Equipment Manufacturing,206


In [20]:
total_by_industry = (
    postings
    .merge(companies_industries, on='company_id', how='left')
    .groupby('industry')
    .size()
    .reset_index(name='total_postings')
)

remote_share_by_industry = (
    remote_by_industry
    .merge(total_by_industry, on='industry')
)

remote_share_by_industry['remote_ratio'] = (
    remote_share_by_industry['num_remote_postings']
    / remote_share_by_industry['total_postings']
)

remote_share_by_industry.sort_values('remote_ratio', ascending=False).head(10)

,industry,num_remote_postings,total_postings,remote_ratio
119,Mobile Gaming Apps,1,1,1.000000
127,Wireless Services,1,1,1.000000
124,Strategic Management Services,1,1,1.000000
14,E-Learning Providers,132,157,0.840764
37,Translation and Localization,52,66,0.787879
49,Market Research,37,48,0.770833
12,Computer and Network Security,176,305,0.577049
36,Professional Training and Coaching,52,91,0.571429
101,Photography,4,7,0.571429
96,Computer Networking Products,5,9,0.555556


This analysis identifies companies with the highest number of job postings that
explicitly allow remote work. Results reflect declared remote availability rather
than a complete remote vs on-site classification.
To account for company size, a remote job ratio was also calculated as the share
of explicitly remote postings over total postings per company.

In [21]:
remote_postings = postings[postings['remote_allowed'] == 1]

remote_by_company = (
    remote_postings
    .groupby(['company_id', 'company_name'])
    .size()
    .reset_index(name='num_remote_postings')
    .sort_values('num_remote_postings', ascending=False)
)

In [22]:
remote_by_company.head(10)

,company_id,company_name,num_remote_postings
4996,73013724.0,J. Galt,604
2572,2204084.0,Talentify.io,276
5780,96139831.0,DataAnnotation,265
5578,90398003.0,Swooped,126
118,3081.0,Thermo Fisher Scientific,97
3053,3651935.0,BetterHelp,96
461,11056.0,Insight Global,85
71,2152.0,TEKsystems,76
321,6849.0,Dice,75
4423,31534680.0,YourEliteMortgage,63


In [24]:
total_by_company = (
    postings
    .groupby(['company_id', 'company_name'])
    .size()
    .reset_index(name='total_postings')
)

remote_ratio_by_company = (
    remote_by_company
    .merge(total_by_company, on=['company_id', 'company_name'])
)

remote_ratio_by_company['remote_ratio'] = (
    remote_ratio_by_company['num_remote_postings']
    / remote_ratio_by_company['total_postings']
)

remote_ratio_by_company.sort_values('remote_ratio', ascending=False).head(10)

,company_id,company_name,num_remote_postings,total_postings,remote_ratio
6132,103456466.0,Foundation Model Startup,1,1,1.0
0,73013724.0,J. Galt,604,604,1.0
1,2204084.0,Talentify.io,276,276,1.0
2,96139831.0,DataAnnotation,265,265,1.0
3,90398003.0,Swooped,126,126,1.0
6114,102273467.0,NovaByte,1,1,1.0
6113,102257756.0,Taxilian Jobs & Hiring,1,1,1.0
6112,103129573.0,GRK Unlimited,1,1,1.0
6111,103126568.0,Elman Tech,1,1,1.0
6110,103120277.0,Dark Alpha Capital LLC,1,1,1.0


## Job demand by industry

Job demand by industry is analyzed using company-level industry classification.
Each job posting is associated with the industry of the hiring company, providing
a stable approximation of industry demand.
While job-level industry classification is available, it was not used in this
analysis to avoid duplication and maintain consistency across sections.

JOB DEMAND
    postings_clean -> job_id, company_id, title

INDUSTRY
    companies_industries -> company_id, industry
    job_industries -> job_id, industry_id
    industries -> industry_id, industry_name

In [26]:
postings_with_industry = postings.merge(
    companies_industries,
    on='company_id',
    how='left'
)

job_demand_by_industry = (
    postings_with_industry
    .groupby('industry')
    .size()
    .reset_index(name='num_postings')
    .sort_values('num_postings', ascending=False)
)

In [27]:
job_demand_by_industry.head(10)

,industry,num_postings
123,Staffing and Recruiting,18886
54,Hospitals and Health Care,15753
56,IT Services and IT Consulting,11573
112,Retail,9642
120,Software Development,5729
40,Financial Services,5516
27,Construction,1969
53,Hospitality,1913
86,Non-profit Organizations,1904
106,Real Estate,1868


## Salary patterns (high-level)

Salary patterns are analyzed at a high level using normalized yearly salaries. Median values are used to reduce the impact of outliers, and only groups with a minimum number of postings are considered to ensure result stability. Industry-level salary patterns reflect the distribution of salaries across companies within each industry and should be interpreted as indicative trends rather than absolute benchmarks.

salaries_clean -> salary_id,job_id,max_salary,med_salary,min_salary,pay_period,currency,compensation_type,salary_value,salary_yearly

postings_clean -> job_id, title

In [29]:
salary_data = salaries.dropna(subset=['salary_yearly'])

salary_with_company = salary_data.merge(
    postings[['job_id', 'company_id', 'company_name']],
    on='job_id',
    how='left'
)
salary_with_company = salary_data.merge(
    postings[['job_id', 'company_id', 'company_name']],
    on='job_id',
    how='left'
)
salary_by_company = (
    salary_with_company
    .groupby(['company_id', 'company_name'])
    .agg(
        median_salary_yearly=('salary_yearly', 'median'),
        num_postings=('salary_yearly', 'count')
    )
    .reset_index()
)
salary_by_company = salary_by_company[
    salary_by_company['num_postings'] >= 5
].sort_values('median_salary_yearly', ascending=False)


In [30]:
salary_by_company.head(10)

,company_id,company_name,median_salary_yearly,num_postings
5210,2379623.0,Interlink Talent Solutions,500000.0,7
971,12686.0,CompHealth,350000.0,7
2202,74477.0,Health eCareers,337000.0,11
9909,92699700.0,Goliath Partners,300000.0,11
6875,10514179.0,DocCafe,275000.0,27
4122,908113.0,Dental Care Alliance,274356.0,25
7221,12979710.0,"Platinum Legal Search Group, LLC",250000.0,9
7709,18577044.0,Provider Solutions & Development,241044.0,5
823,9814.0,Eastridge Workforce Solutions,225000.0,5
2670,145675.0,Marina Sirras & Associates LLC,225000.0,5


In [31]:
salary_with_industry = salary_with_company.merge(
    companies_industries,
    on='company_id',
    how='left'
)

salary_by_industry = (
    salary_with_industry
    .groupby('industry')
    .agg(
        median_salary_yearly=('salary_yearly', 'median'),
        num_postings=('salary_yearly', 'count')
    )
    .reset_index()
)

salary_by_industry = salary_by_industry[
    salary_by_industry['num_postings'] >= 20
].sort_values('median_salary_yearly', ascending=False)


In [32]:
salary_by_industry.head(10)

,industry,median_salary_yearly,num_postings
21,Computer Hardware Manufacturing,122687.5,30
23,Computer and Network Security,117000.0,109
24,Computers and Electronics Manufacturing,113500.0,40
115,Semiconductor Manufacturing,113100.0,86
13,Biotechnology Research,110000.0,313
117,Software Development,110000.0,1975
121,"Technology, Information and Internet",110000.0,89
106,Renewable Energy Semiconductor Manufacturing,105200.0,97
85,Online Audio and Video Media,100000.0,38
66,Legal Services,100000.0,25


To explore whether company size is associated with higher salaries, company-level
median salaries were compared against employee counts. Company size is used as a
proxy based on available employee count snapshots. Results indicate general
trends rather than causal relationships. Companies were grouped into size quartiles to improve interpretability of salary
patterns across different company sizes.

In [37]:
salary_company = salaries.dropna(subset=['salary_yearly']).merge(
    postings[['job_id', 'company_id', 'company_name']],
    on='job_id',
    how='left'
)

salary_by_company = (
    salary_company
    .groupby(['company_id', 'company_name'])
    .agg(
        median_salary_yearly=('salary_yearly', 'median'),
        num_salary_records=('salary_yearly', 'count')
    )
    .reset_index()
)

salary_by_company = salary_by_company[
    salary_by_company['num_salary_records'] >= 5
]

company_salary_size = salary_by_company.merge(
    employee_counts[['company_id', 'employee_count']],
    on='company_id',
    how='left'
)

company_salary_size = company_salary_size.dropna(subset=['employee_count'])

company_salary_size['company_size_group'] = pd.qcut(
    company_salary_size['employee_count'],
    q=4,
    labels=['Small', 'Medium', 'Large', 'Very Large']
)

salary_by_size_group = (
    company_salary_size
    .groupby('company_size_group', observed=True)
    .agg(
        median_salary=('median_salary_yearly', 'median'),
        companies=('company_id', 'nunique')
    )
    .reset_index()
)


salary_by_size_group

,company_size_group,median_salary,companies
0,Small,83200.0,549
1,Medium,65000.0,331
2,Large,57500.0,280
3,Very Large,73610.0,220
